# Can we get `timeActual` for historical NBA PBP?

The CDN live PBP (`cdn.nba.com`) includes `timeActual` (wall-clock ISO timestamp per action),
but the stats API (`stats.nba.com` via `PlayByPlayV3`) does not.

**Result:** Yes — the CDN endpoint serves historical PBP with `timeActual` for any game ID,
not just today's games. We have since deprecated `nba_stats` and consolidated all NBA data
on `cdn.nba.com`. See the cells below for the original investigation.

## Endpoints in `nba_api`

| Endpoint | Package | Source | Has `timeActual`? |
|----------|---------|--------|-------------------|
| `PlayByPlayV3` | `nba_api.stats.endpoints` | `stats.nba.com` | No (24 fields) |
| `PlayByPlayV2` | `nba_api.stats.endpoints` | `stats.nba.com` | No (deprecated, had `WCTIMESTRING`) |
| `PlayByPlay` | `nba_api.live.nba.endpoints` | `cdn.nba.com` | **Yes** |

## Questions answered

1. Does `cdn.nba.com` still serve PBP for old game IDs? **Yes**
2. Does `PlayByPlayV2` still work? **No** (deprecated, returns empty JSON)
3. Can we hit the CDN REST endpoint directly for a historical game? **Yes, with `timeActual`**

In [ ]:
import json
import boto3
import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 80)

S3_BUCKET = "prediction-markets-data"
s3 = boto3.client("s3")

# Load schedule to get sample game IDs
obj = s3.get_object(Bucket=S3_BUCKET, Key="nba_cdn/schedule/season_2025-26.json")
schedule_raw = json.loads(obj["Body"].read())

# Extract completed games
completed_games = [
    g
    for d in schedule_raw["leagueSchedule"]["gameDates"]
    for g in d["games"]
    if g["gameStatus"] == 3
]
print(f"{len(completed_games)} completed games loaded")

# Pick sample game IDs at different points in the season
n = len(completed_games)
sample_indices = [0, n // 2, n - 1]
sample_ids = [completed_games[i]["gameId"] for i in sample_indices]
for i in sample_indices:
    g = completed_games[i]
    away = g["awayTeam"]["teamTricode"]
    home = g["homeTeam"]["teamTricode"]
    print(f"  {g['gameId']}  {g['gameDateEst'][:10]}  {away} @ {home}")

## 1. Try `cdn.nba.com` REST directly for historical games

The live PBP endpoint is:
```
https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json
```

Let's see if it still serves data for old game IDs.

In [ ]:
CDN_HEADERS = {
    "Accept": "application/json",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

for i, game_id in enumerate(sample_ids):
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
    resp = requests.get(url, headers=CDN_HEADERS)
    g = completed_games[sample_indices[i]]
    away = g["awayTeam"]["teamTricode"]
    home = g["homeTeam"]["teamTricode"]
    label = f"{game_id} ({g['gameDateEst'][:10]}, {away} @ {home})"
    print(f"{label}: HTTP {resp.status_code}", end="")
    if resp.status_code == 200:
        data = resp.json()
        actions = data.get("game", {}).get("actions", [])
        has_timeactual = any("timeActual" in a for a in actions[:5])
        print(f"  -> {len(actions)} actions, timeActual={'YES' if has_timeactual else 'NO'}")
        if has_timeactual:
            print(f"     Sample: {actions[1].get('timeActual', 'N/A')}")
    else:
        print()

## 2. Try `nba_api.live.nba.endpoints.PlayByPlay` for historical games

This is the SDK wrapper around the same CDN endpoint. Let's see if it works for old games.

In [ ]:
from nba_api.live.nba.endpoints import playbyplay as live_pbp

game_id = sample_ids[0]
g = completed_games[sample_indices[0]]
away = g["awayTeam"]["teamTricode"]
home = g["homeTeam"]["teamTricode"]
print(f"Trying nba_api live PlayByPlay for {game_id} ({g['gameDateEst'][:10]} {away} @ {home})")

try:
    pbp = live_pbp.PlayByPlay(game_id)
    actions = pbp.get_dict()["game"]["actions"]
    print(f"{len(actions)} actions")
    print(f"Fields in first action: {list(actions[0].keys())}")
    if "timeActual" in actions[0]:
        print(f"\ntimeActual present! Sample values:")
        for a in actions[:5]:
            print(f"  action {a['actionNumber']:3d}  period {a['period']}  clock {a['clock']}  timeActual {a.get('timeActual', 'N/A')}")
    else:
        print("\ntimeActual NOT present in response")
except Exception as e:
    print(f"Error: {e}")

## 3. Try `PlayByPlayV2` (deprecated stats.nba.com endpoint)

PlayByPlayV2 had a `WCTIMESTRING` field (wall-clock time string). It's deprecated but may still work.

In [ ]:
from nba_api.stats.endpoints import playbyplayv2

game_id = sample_ids[0]
g = completed_games[sample_indices[0]]
away = g["awayTeam"]["teamTricode"]
home = g["homeTeam"]["teamTricode"]
print(f"Trying PlayByPlayV2 for {game_id} ({g['gameDateEst'][:10]} {away} @ {home})")

try:
    pbp_v2 = playbyplayv2.PlayByPlayV2(game_id=game_id)
    df_v2 = pbp_v2.get_data_frames()[0]
    print(f"{len(df_v2)} rows, {df_v2.shape[1]} columns")
    print(f"\nColumns: {df_v2.columns.tolist()}")
    # Check for wall-clock fields
    wc_cols = [c for c in df_v2.columns if "wc" in c.lower() or "time" in c.lower() or "actual" in c.lower()]
    print(f"\nTime-related columns: {wc_cols}")
    if wc_cols:
        print(f"\nSample values:")
        print(df_v2[wc_cols].head(10))
except Exception as e:
    print(f"Error: {e}")

## 4. Compare field coverage

Side-by-side comparison of what each endpoint returns.

In [ ]:
from nba_api.stats.endpoints import playbyplayv3

game_id = sample_ids[0]

# PlayByPlayV3 (stats.nba.com)
pbp_v3 = playbyplayv3.PlayByPlayV3(game_id=game_id, start_period=1, end_period=10)
v3_df = pbp_v3.get_data_frames()[0]

# CDN REST (cdn.nba.com)
url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
resp = requests.get(url, headers=CDN_HEADERS)
cdn_df = None
if resp.status_code == 200:
    cdn_actions = resp.json().get("game", {}).get("actions", [])
    cdn_df = pd.DataFrame(cdn_actions)

print(f"PlayByPlayV3 (stats.nba.com): {v3_df.shape[1]} columns, {len(v3_df)} rows")
if cdn_df is not None:
    print(f"CDN REST (cdn.nba.com):       {cdn_df.shape[1]} columns, {len(cdn_df)} rows")
    print(f"\n--- V3 only ---")
    print(sorted(set(v3_df.columns) - set(cdn_df.columns)))
    print(f"\n--- CDN only ---")
    print(sorted(set(cdn_df.columns) - set(v3_df.columns)))
    print(f"\n--- Common ---")
    print(sorted(set(v3_df.columns) & set(cdn_df.columns)))
else:
    print("CDN REST: not available for this game")
    print(f"\nV3 columns: {sorted(v3_df.columns.tolist())}")

In [6]:
# If CDN worked, show the timeActual column alongside game clock
if cdn_df is not None and "timeActual" in cdn_df.columns:
    print("CDN PBP with timeActual (first 15 scoring actions):")
    scoring = cdn_df[cdn_df["actionType"].isin(["2pt", "3pt", "Made Shot"])].head(15)
    print(scoring[["actionNumber", "period", "clock", "timeActual", "description", "scoreHome", "scoreAway"]].to_string())
else:
    print("CDN data not available or timeActual not present")

CDN PBP with timeActual (first 15 scoring actions):
    actionNumber  period        clock              timeActual                                                   description scoreHome scoreAway
2              7       1  PT11M48.00S  2024-10-22T23:36:07.5Z              J. Tatum 26' 3PT pullup (3 PTS) (D. White 1 AST)         3         0
3              9       1  PT11M27.00S  2024-10-22T23:36:28.1Z           O. Anunoby driving Layup (2 PTS) (J. Brunson 1 AST)         3         2
4             11       1  PT11M16.00S  2024-10-22T23:36:38.5Z                                  MISS J. Tatum 25' pullup 3PT         3         2
6             13       1  PT11M09.00S  2024-10-22T23:36:46.6Z                              J. Brunson running Layup (2 PTS)         3         4
7             14       1  PT10M54.00S  2024-10-22T23:37:01.7Z  D. White driving floating Jump Shot (2 PTS) (J. Tatum 1 AST)         5         4
8             16       1  PT10M35.00S  2024-10-22T23:37:19.6Z                       

## 5. Summary

Based on the results above, determine the best path for getting `timeActual` on historical PBP data.